In [1]:
"""
Google Trends Daily Collection Script
======================================
Designed for 2-hour daily sessions. Run it, let it go for 2 hours,
stop it (Ctrl+C), and pick up tomorrow where you left off.

SETUP:
    pip install pytrends pandas numpy

USAGE:
    1. Each person gets their own batch file:
       - Person A runs with: batch_person_A.csv
       - Person B runs with: batch_person_B.csv

    2. Edit INPUT_CSV below to point to YOUR batch file

    3. Run:  python collect_daily.py

    4. After ~2 hours, stop with Ctrl+C (progress is auto-saved)

    5. Tomorrow, run the same command. It automatically skips
       books you already collected.

SCHEDULE:
    ~175 books per 2-hour session
    Person A: 3,718 titles → ~22 sessions
    Person B: 3,718 titles → ~22 sessions
    Running daily: done in ~22 days
"""


'\nGoogle Trends Daily Collection Script\n======================================\nDesigned for 2-hour daily sessions. Run it, let it go for 2 hours,\nstop it (Ctrl+C), and pick up tomorrow where you left off.\n\nSETUP:\n    pip install pytrends pandas numpy\n\nUSAGE:\n    1. Each person gets their own batch file:\n       - Person A runs with: batch_person_A.csv\n       - Person B runs with: batch_person_B.csv\n\n    2. Edit INPUT_CSV below to point to YOUR batch file\n\n    3. Run:  python collect_daily.py\n\n    4. After ~2 hours, stop with Ctrl+C (progress is auto-saved)\n\n    5. Tomorrow, run the same command. It automatically skips\n       books you already collected.\n\nSCHEDULE:\n    ~175 books per 2-hour session\n    Person A: 3,718 titles → ~22 sessions\n    Person B: 3,718 titles → ~22 sessions\n    Running daily: done in ~22 days\n'

In [2]:
import pandas as pd
import numpy as np
from pytrends.request import TrendReq
import time
from datetime import datetime, timedelta
import os
import signal
import sys

# ============================================================
# CONFIGURATION
# ============================================================

INPUT_CSV = 'batch_person_B.csv'   

OUTPUT_CSV = 'my_google_trends_results.csv'  # Your results accumulate here

SLEEP_BETWEEN_CALLS = 10    # Seconds between before/after API calls
SLEEP_BETWEEN_BOOKS = 15    # Seconds between books
SLEEP_ON_RATE_LIMIT = 300   # 5 minutes on rate limit
MAX_RETRIES = 3


results = []
shutting_down = False

def save_and_exit(signum=None, frame=None):
    global shutting_down
    if shutting_down:
        sys.exit(1)
    shutting_down = True
    print(f"\n\n💾 Saving {len(results)} results to {OUTPUT_CSV}...")
    if results:
        pd.DataFrame(results).to_csv(OUTPUT_CSV, index=False)
        print(f"✓ Saved! You collected {len(results)} books total.")
        print(f"  Run this script again tomorrow to continue.")
    sys.exit(0)

signal.signal(signal.SIGINT, save_and_exit)
signal.signal(signal.SIGTERM, save_and_exit)

<Handlers.SIG_DFL: 0>

In [3]:
# ============================================================
# COLLECTION FUNCTION
# ============================================================

pytrends = TrendReq(hl='en-US', tz=360)

def get_trends_data(book_title, ban_date_str):
    try:
        ban_date = datetime.strptime(ban_date_str, '%Y-%m-%d')
    except Exception:
        return make_result(book_title, ban_date_str, success=False, error="Invalid date")

    before_tf = f"{(ban_date - timedelta(days=90)):%Y-%m-%d} {(ban_date - timedelta(days=1)):%Y-%m-%d}"
    after_tf = f"{(ban_date + timedelta(days=1)):%Y-%m-%d} {(ban_date + timedelta(days=90)):%Y-%m-%d}"

    for attempt in range(MAX_RETRIES):
        try:
            # BEFORE
            pytrends.build_payload([book_title], timeframe=before_tf, geo='US')
            before_df = pytrends.interest_over_time()
            if not before_df.empty and book_title in before_df.columns:
                bv = before_df[book_title].values
                b_avg, b_max, b_min = float(np.mean(bv)), float(np.max(bv)), float(np.min(bv))
            else:
                bv = np.array([])
                b_avg = b_max = b_min = 0.0

            time.sleep(SLEEP_BETWEEN_CALLS)

            # AFTER
            pytrends.build_payload([book_title], timeframe=after_tf, geo='US')
            after_df = pytrends.interest_over_time()
            if not after_df.empty and book_title in after_df.columns:
                av = after_df[book_title].values
                a_avg, a_max, a_min = float(np.mean(av)), float(np.max(av)), float(np.min(av))
            else:
                av = np.array([])
                a_avg = a_max = a_min = 0.0

            # Metrics
            if b_avg > 0:
                pct = ((a_avg - b_avg) / b_avg) * 100
            else:
                pct = 0.0 if a_avg == 0 else float('inf')

            all_v = np.concatenate([bv, av]) if len(bv) + len(av) > 0 else np.array([])
            vol = float(np.std(all_v)) if len(all_v) > 0 else 0.0

            return make_result(book_title, ban_date_str, success=True,
                             b_avg=b_avg, a_avg=a_avg, b_max=b_max, a_max=a_max,
                             b_min=b_min, a_min=a_min, pct=pct,
                             abs_change=a_avg - b_avg, vol=vol)

        except Exception as e:
            err = str(e)
            if '429' in err or 'Too Many' in err or 'quota' in err.lower():
                wait = SLEEP_ON_RATE_LIMIT * (attempt + 1)
                print(f"\n  ⚠ Rate limited! Waiting {wait//60} min...")
                time.sleep(wait)
            else:
                if attempt == MAX_RETRIES - 1:
                    return make_result(book_title, ban_date_str, success=False, error=err[:100])
                time.sleep(20)

    return make_result(book_title, ban_date_str, success=False, error="Retries exhausted")


def make_result(title, date, success, b_avg=None, a_avg=None, b_max=None,
                a_max=None, b_min=None, a_min=None, pct=None,
                abs_change=None, vol=None, error=''):
    return {
        'book_title': title,
        'ban_date': date,
        'avg_search_before': round(b_avg, 2) if b_avg is not None else None,
        'avg_search_after': round(a_avg, 2) if a_avg is not None else None,
        'max_search_before': b_max,
        'max_search_after': a_max,
        'min_search_before': b_min,
        'min_search_after': a_min,
        'percent_change': round(pct, 2) if pct is not None else None,
        'absolute_change': round(abs_change, 2) if abs_change is not None else None,
        'volatility': round(vol, 2) if vol is not None else None,
        'data_collected': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'collection_successful': success,
        'state': '',
        'error': error
    }


In [4]:


# ============================================================
# MAIN
# ============================================================

def main():
    global results

    batch = pd.read_csv(INPUT_CSV)
    print(f"Batch file: {INPUT_CSV} ({len(batch)} titles)")

    # Load previous progress
    done_titles = set()
    if os.path.exists(OUTPUT_CSV):
        prev = pd.read_csv(OUTPUT_CSV)
        results = prev.to_dict('records')
        done_titles = set(prev[prev['collection_successful'] == True]['book_title'].str.strip().str.lower())
        print(f"Previous progress: {len(done_titles)} titles already collected")

    remaining = batch[~batch['book_title'].str.strip().str.lower().isin(done_titles)]
    print(f"Remaining this session: {len(remaining)} titles")
    print(f"Estimated time for all remaining: {len(remaining) * 35 / 3600:.1f} hours")
    print(f"\nPress Ctrl+C at any time to save and stop.\n")
    print(f"{'─'*70}")

    session_count = 0
    session_start = time.time()

    for idx, row in remaining.iterrows():
        title = row['book_title']
        ban_date = row['ban_date']
        state = row.get('state', '')

        elapsed = (time.time() - session_start) / 60
        pos = len(done_titles) + session_count + 1
        total = len(batch)
        print(f"[{pos}/{total}] ({elapsed:.0f}min) {title[:50]}...", end=' ', flush=True)

        result = get_trends_data(title, ban_date)
        result['state'] = state
        results.append(result)
        session_count += 1

        if result['collection_successful']:
            b = result['avg_search_before']
            a = result['avg_search_after']
            print(f"✓ ({b} → {a})")
        else:
            print(f"✗ ({result['error'][:35]})")

        # Auto-save every 25 books
        if session_count % 25 == 0:
            pd.DataFrame(results).to_csv(OUTPUT_CSV, index=False)
            elapsed_h = elapsed / 60
            rate = session_count / max(elapsed, 1) * 60  # books per hour
            left = len(remaining) - session_count
            eta_h = left / max(rate, 1) / 60
            print(f"  💾 Saved ({session_count} new today, {len(results)} total, ~{rate:.0f} books/hr, ~{eta_h:.1f}h remaining)")

        time.sleep(SLEEP_BETWEEN_BOOKS)

    # Done with all books
    pd.DataFrame(results).to_csv(OUTPUT_CSV, index=False)
    print(f"\n{'='*70}")
    print(f"ALL DONE! Collected {session_count} new books this session.")
    print(f"Total results: {len(results)}")
    print(f"Saved to: {OUTPUT_CSV}")


if __name__ == '__main__':
    main()

Batch file: batch_person_B.csv (3718 titles)
Previous progress: 1574 titles already collected
Remaining this session: 2137 titles
Estimated time for all remaining: 20.8 hours

Press Ctrl+C at any time to save and stop.

──────────────────────────────────────────────────────────────────────
[1575/3718] (0min) 100 Questions You'd Never Ask Your Parents: Straig... 
  ⚠ Rate limited! Waiting 5 min...


KeyboardInterrupt: 